# English control · Llama-3.2-3B-Instruct, end to end

The four stages of the Hindi study, run in one notebook on the **same prompts**, so English and Hindi can be compared:

1. **Generate:** `llama3.2:3b-instruct-q8_0` answers the exact prompts of our annotated Hindi answers, in English. The prompts are matched by their Hindi text in `shreshthamodi02/longfact-hindi-paper`, whose `prompt_en` is the paper's exact English prompt. Generation goes through Ollama in 8-bit with the paper's settings (temperature 0.1, up to 2,048 tokens, single turn), the same way the Hindi answers were written.
2. **Annotate:** the same web-search judge as the Hindi annotations: the paper repo's prompt and code, gpt-oss-120b on HF credits, Exa MCP search with DuckDuckGo as fallback. In English the paper's prompt is used verbatim, with no Hindi note.
3. **Train probes** on the fp16 model's activations: the same detectors, loss, 2-fold cross-validation and settings as the Hindi probe notebooks.
4. **Run the held-out answers through the probes** and compare what lights up with what the judge tagged.
5. **Compare with Hindi:** how often the judge flags hallucinations on the shared prompts, and how well the probes find them, layer by layer.

The paper repo has no attention probe, only the linear probe (`ValueHeadProbe`) and LoRA. It released no probe for Llama-3.2-3B.
The attention-output and per-head probes here are ours, as in the Hindi notebooks.

Set `LANG = "hi"` to run the same prompts in Hindi with this model instead, for a within-model English vs Hindi comparison.

**Saving:** answers are uploaded every 20, annotations every 5, probe checkpoints every 25 texts, and each finished fold is uploaded.
If the session stops, Run all again and finished work is skipped.

**Kaggle:** GPU **T4** (x1 or x2), Internet **on**, Add-ons → Secrets → **`HF_TOKEN`**: the shreshtha2002 token (write + "Make calls to Inference Providers").
For the probes, that account needs access to `meta-llama/Llama-3.2-3B-Instruct`. Llama 3.2 has its own licence, separate from 3.1.
Without access, the notebook loads `unsloth/Llama-3.2-3B-Instruct`, an ungated copy with the same weight files.

**Time / cost:** about 15 min to generate, 30–60 min to annotate (about $0.05 per answer, roughly $3), and about 1 h for the probes.

In [ ]:
import os, shutil, subprocess, sys
subprocess.run("apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null", shell=True)   # Ollama's installer needs zstd
if not shutil.which("ollama"):
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score", "nltk", "absl-py", "ddgs", "openai", "pydantic>=2.6",
                "peft", "jaxtyping", "termcolor", "circuitsvis"], check=True)
if not os.path.isdir("repo"):     # the paper's code
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/obalcells/hallucination_probes", "repo"], check=True)
assert shutil.which("ollama"), "Ollama did not install: read the output above (is Internet on?)"
print("ready")

In [ ]:
# ---------------- settings ----------------
import re
LANG = "en"                                       # "en" = English control; "hi" = the same prompts in Hindi with this model
OLLAMA_TAG = "llama3.2:3b-instruct-q8_0"          # writes the answers (Ollama, 8-bit), as for the Hindi answers
MODEL = "meta-llama/Llama-3.2-3B-Instruct"        # the same model in fp16, read by the probes
MODEL_FALLBACK = "unsloth/Llama-3.2-3B-Instruct"  # used only if MODEL is gated for this account: same weight files, no licence click
HF_USER = "shreshtha2002"                         # receives every output; HF_TOKEN is its write token
PROMPTS_OF = "shreshtha2002/longfact-hi-llama3.1-8b-instruct-q8-0-annotated"   # use exactly the prompts of these annotated Hindi answers
PROMPT_SRC = "shreshthamodi02/longfact-hindi-paper"   # each prompt in Hindi (prompt_hi) and the paper's exact English (prompt_en)
SLUG = re.sub(r"[^a-zA-Z0-9.]+", "-", OLLAMA_TAG).strip("-").lower()             # llama3.2-3b-instruct-q8-0, as in the other dataset names
GEN_REPO = f"{HF_USER}/longfact-{LANG}-{SLUG}"    # stage 1 output
ANN_REPO = f"{GEN_REPO}-annotated"                # stage 2 output

# stage 1: generation, the paper's settings (App. C), the same as the Hindi answers
TEMPERATURE, MAX_NEW_TOKENS, NUM_CTX = 0.1, 2048, 4096
GEN_WORKERS, CHUNK = 4, 20                        # parallel answers; answers per uploaded file
# stage 2: annotation, the same as the Hindi annotations (repo annotate.py / run.py)
JUDGE = "openai/gpt-oss-120b:groq"
JUDGE_TEMPERATURE, JUDGE_MAX_TOKENS, MAX_SEARCHES = 0.0, 8192, 15
EXA_MCP = "https://mcp.exa.ai/mcp?tools=web_search_exa"   # Exa's hosted MCP server, keyless with a free rate limit
ANN_WORKERS, SAVE_EVERY = 4, 5                    # answers judged in parallel; upload progress every 5 answers
HINDI_NOTE = LANG == "hi"                         # the Hindi annotations added one note to the paper's prompt; English uses it verbatim

# stages 3-4: probes, the same settings as the Hindi probe notebooks
SETTING = "control"
PAPER_PROBE = None                                # the paper released no probe for Llama-3.2-3B
OUT_REPO, RUN = f"{HF_USER}/hallucination-probes-multiturn", f"control-{LANG}-{SLUG}"   # results: OUT_REPO/RUN/
N_PAPER_TRAIN = 0                                 # ours only, as in Hindi
FOLDS = 2                                         # cross-validation folds over our answers (by prompt)
EPOCHS = 10                                       # passes over the training texts
MAX_LEN = 2560                                    # tokens per text
POS_WEIGHT = NEG_WEIGHT = 10.0                    # repo train_config: entity tokens weigh 10, other answer tokens 1
LR, BATCH = 5e-4, 8                               # released llama3_1_8b_linear training_config (probe_head_lr, batch)
MLP_HIDDEN = 64
CACHE_PAPER = 50                                  # unused here (no paper texts)
CKPT_EVERY = 25                                   # texts between checkpoints
ATTN_EXAMPLES, ATTN_WINDOW = 8, 1024              # head / attention analysis: answers and tokens per window
SEED = 42

# stage 5: what to compare with (already on the Hub)
COMPARE_ANN = {"Hindi · Llama-3.1-8B": "shreshtha2002/longfact-hi-llama3.1-8b-instruct-q8-0-annotated",
               "Hindi · Aya Expanse 8B": "shreshtha2002/longfact-hi-aya-expanse-8b-q8-0-annotated"}
COMPARE_RUNS = {"Hindi · Llama-3.1-8B": "hindi-llama", "Hindi · Aya Expanse 8B": "hindi-aya"}   # probe results in OUT_REPO

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation (set before torch is imported)
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "out"
WORK = f"{OUT}/work"
os.makedirs(f"{OUT}/figs", exist_ok=True); os.makedirs(WORK, exist_ok=True)
THIS = f"{'English' if LANG == 'en' else 'Hindi'} · Llama-3.2-3B"   # this run's name in the comparison

In [ ]:
# ---------------- reused code: the paper's repo + ARENA 1.3.1 ----------------
import gc, html as htmllib, json, math, random, re, time
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from IPython.display import HTML, display
sys.path.insert(0, "repo")
from probe.loss import compute_probe_bce_loss, compute_probe_max_aggregation_loss   # paper's token BCE + span-max losses
from probe.dataset_converters import _MAP_LABEL_TO_SCALAR                          # NS / II -> 1, S -> 0
from utils.tokenization import find_string_in_tokens                                # paper's span -> token search
from utils.metrics import compute_clf_metrics                                       # paper's metrics
t = torch

# ---- ARENA 1.3.1 solutions, verbatim ----
def get_pca_components(activations, k=2):
    X = activations - activations.mean(dim=0)
    cov = X.t() @ X / (X.shape[0] - 1)
    eigenvalues, eigenvectors = t.linalg.eigh(cov)
    sorted_indices = t.argsort(eigenvalues, descending=True)
    return eigenvectors[:, sorted_indices[:k]]

class MMProbe(t.nn.Module):
    def __init__(self, direction, covariance=None, atol=1e-3, bias=None):
        super().__init__()
        self.direction = t.nn.Parameter(direction, requires_grad=False)
        self.bias = t.nn.Parameter(bias, requires_grad=False) if bias is not None else None
        self.inv = t.nn.Parameter(t.linalg.pinv(covariance, hermitian=True, atol=atol), requires_grad=False) if covariance is not None else None
    def forward(self, x, iid=False):
        if self.bias is not None:
            x = x - self.bias
        return t.sigmoid(x @ self.inv @ self.direction) if (iid and self.inv is not None) else t.sigmoid(x @ self.direction)
    @staticmethod
    def from_data(acts, labels, device="cpu", bias=True):
        acts, labels = acts.to(device), labels.to(device)
        pos_acts, neg_acts = acts[labels == 1], acts[labels == 0]
        pos_mean, neg_mean = pos_acts.mean(0), neg_acts.mean(0)
        centered = t.cat([pos_acts - pos_mean, neg_acts - neg_mean], dim=0)
        covariance = centered.t() @ centered / acts.shape[0]
        return MMProbe(pos_mean - neg_mean, covariance=covariance, bias=0.5 * (pos_mean + neg_mean) if bias else None).to(device)

class LRProbe(t.nn.Module):
    def __init__(self, d_in, scaler_mean=None, scaler_scale=None):
        super().__init__()
        self.net = t.nn.Sequential(t.nn.Linear(d_in, 1, bias=False), t.nn.Sigmoid())
        self.register_buffer("scaler_mean", scaler_mean); self.register_buffer("scaler_scale", scaler_scale)
    def forward(self, x):
        if self.scaler_mean is not None:
            x = (x - self.scaler_mean) / self.scaler_scale
        return self.net(x).squeeze(-1)
    @property
    def direction(self):
        return self.net[0].weight.data[0]
    @staticmethod
    def from_data(acts, labels, C=0.1, device="cpu"):
        X, y = acts.cpu().float().numpy(), labels.cpu().float().numpy()
        scaler = StandardScaler(); X_scaled = scaler.fit_transform(X)
        lr_model = LogisticRegression(C=C, random_state=42, fit_intercept=False, max_iter=1000).fit(X_scaled, y)
        probe = LRProbe(acts.shape[-1], t.tensor(scaler.mean_, dtype=t.float32), t.tensor(scaler.scale_, dtype=t.float32)).to(device)
        probe.net[0].weight.data[0] = t.tensor(lr_model.coef_[0], dtype=t.float32).to(device)
        return probe
log = lambda m: print(time.strftime("%H:%M:%S"), m, flush=True)

In [ ]:
# ---------------- stage 1: answer the Hindi study's prompts with OLLAMA_TAG ----------------
import threading, urllib.request
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import HfApi, hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "No HF token: Kaggle > Add-ons > Secrets > add HF_TOKEN and tick it for this notebook"
api = HfApi(token=HF_TOKEN)
me = api.whoami()
assert HF_USER in [me["name"]] + [o["name"] for o in me.get("orgs", [])], f"HF_TOKEN belongs to '{me['name']}', not '{HF_USER}'"
log(f"HF login: {me['name']}")

# the prompts: those of the annotated Hindi answers, matched to longfact-hindi-paper by their exact Hindi text
hindi = [json.loads(l) for l in open(hf_hub_download(PROMPTS_OF, "progress/progress.jsonl", repo_type="dataset", token=HF_TOKEN), encoding="utf-8")]
hindi = list({r["prompt_id"]: r for r in hindi if r.get("annotations") is not None}.values())
src = {}
for short in ("longfact", "longfact-augmented"):
    for l in open(hf_hub_download(PROMPT_SRC, f"data/test-{short}.jsonl", repo_type="dataset", token=HF_TOKEN), encoding="utf-8"):
        p = json.loads(l); src[p["prompt_hi"].strip()] = p
prompts, missing = [], []
for r in hindi:
    p = src.get(next(m["content"] for m in r["conversation"] if m["role"] == "user").strip())
    if p is None:
        missing.append(r["prompt_id"]); continue
    prompts.append({"prompt_id": r["prompt_id"], "subset": p["subset"], "canary": p["canary"], "split": p["split"],
                    "prompt": p["prompt_en"] if LANG == "en" else p["prompt_hi"]})
assert not missing, f"{len(missing)} Hindi prompts not found in {PROMPT_SRC}: {missing[:5]}"
log(f"{len(prompts)} prompts (the same as {PROMPTS_OF}), e.g. {prompts[0]['prompt'][:150]!r}")

def chat(messages):
    body = {"model": OLLAMA_TAG, "messages": messages, "stream": False, "keep_alive": "60m",
            "options": {"temperature": TEMPERATURE, "num_predict": MAX_NEW_TOKENS, "num_ctx": NUM_CTX}}
    req = urllib.request.Request("http://127.0.0.1:11434/api/chat", data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=1800))["message"]["content"].strip()

def answer(p):
    conv = [{"role": "user", "content": p["prompt"]}]
    conv.append({"role": "assistant", "content": chat(conv)})
    return {"subset": p["subset"], "model": MODEL, "conversation": conv, "canary": p["canary"], "split": p["split"],
            "lang": LANG, "prompt_id": p["prompt_id"], "ollama_tag": OLLAMA_TAG}

api.create_repo(GEN_REPO, repo_type="dataset", private=False, exist_ok=True)
on_hub = set(api.list_repo_files(GEN_REPO, repo_type="dataset"))
chunks = [(f"data/{c // CHUNK:05d}.jsonl", prompts[c:c + CHUNK]) for c in range(0, len(prompts), CHUNK)]
server = None
if any(not os.path.exists(f"{WORK}/gen/{n}") and n not in on_hub for n, _ in chunks):
    server = subprocess.Popen([shutil.which("ollama"), "serve"], env=dict(os.environ, OLLAMA_NUM_PARALLEL=str(GEN_WORKERS)),
                              stdout=open(f"{WORK}/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434", timeout=5); break
        except Exception:
            time.sleep(2)
    subprocess.run([shutil.which("ollama"), "pull", OLLAMA_TAG], check=True, capture_output=True)
answers = []
for name, ps in chunks:
    local = f"{WORK}/gen/{name}"
    if not os.path.exists(local):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        if name in on_hub:
            shutil.copy(hf_hub_download(GEN_REPO, name, repo_type="dataset", token=HF_TOKEN), local)
        else:
            tic = time.time()
            with ThreadPoolExecutor(GEN_WORKERS) as pool:
                rows = list(pool.map(answer, ps))
            with open(local, "w", encoding="utf-8") as f:
                f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)
            try:
                api.upload_file(path_or_fileobj=local, path_in_repo=name, repo_id=GEN_REPO, repo_type="dataset", commit_message=f"add {name}")
            except Exception as ex:
                log(f"  upload failed (kept locally): {str(ex)[:100]}")
            log(f"{name}: {len(rows)} answers in {time.time() - tic:.0f}s")
    answers += [json.loads(l) for l in open(local, encoding="utf-8")]
if server is not None:                                   # free the GPU for the probes
    server.terminate(); server.wait()
    subprocess.run("pkill -f ollama || true", shell=True)
    if torch.cuda.is_available():
        log(f"GPU 0 free after stopping Ollama: {torch.cuda.mem_get_info(0)[0] / 2**30:.1f} GiB")
log(f"stage 1 done: {len(answers)} answers -> https://huggingface.co/datasets/{GEN_REPO}")
print(answers[0]["conversation"][-1]["content"][:400])

In [ ]:
# ---------------- stage 2: annotate every answer with the web-search judge (the repo's prompt and code) ----------------
import ast, importlib.util, logging
from typing import List
from ddgs import DDGS
from openai import OpenAI
from rouge_score import rouge_scorer
from utils.parsing import parse_and_validate_json
import utils.string_utils as string_utils
from utils.string_utils import try_matching_span_in_text

spec = importlib.util.spec_from_file_location("data_models", "repo/annotation_pipeline/data_models.py")
data_models = importlib.util.module_from_spec(spec); spec.loader.exec_module(data_models)
AnnotatedSpan = data_models.AnnotatedSpan
# format_prompt and assign_span_positions, taken verbatim from annotate.py (importing the file itself would need Anthropic's safety-tooling)
repo_fns = {"List": List, "AnnotatedSpan": AnnotatedSpan, "try_matching_span_in_text": try_matching_span_in_text,
            "logger": logging.getLogger("annotate")}
for node in ast.parse(open("repo/annotation_pipeline/annotate.py", encoding="utf-8").read()).body:
    if isinstance(node, ast.FunctionDef) and node.name in ("format_prompt", "assign_span_positions"):
        exec(compile(ast.Module([node], []), "annotate.py", "exec"), repo_fns)
format_prompt, assign_span_positions = repo_fns["format_prompt"], repo_fns["assign_span_positions"]
PROMPT = open("repo/annotation_pipeline/entity_annotation.prompt", encoding="utf-8").read().strip()   # as in annotate.py

if LANG == "hi":                 # ROUGE's default tokenizer deletes non-Latin text (as in the Hindi annotation notebooks)
    class UnicodeTokenizer:
        def tokenize(self, text):
            return [w for w in re.split(r"[\s.,;:!?\"'()\[\]\-‐‑‒–—―।]+", text.lower()) if w]
    string_utils.ROUGE_SCORER = rouge_scorer.RougeScorer(["rougeL"], tokenizer=UnicodeTokenizer())

client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=HF_TOKEN, max_retries=6, timeout=600)

def exa_mcp(query):
    # minimal MCP (streamable HTTP) client: initialize -> initialized -> tools/call web_search_exa
    H = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream",
         "User-Agent": "Mozilla/5.0 (hallucination-annotator)"}
    def rpc(body, sid=None):
        req = urllib.request.Request(EXA_MCP, data=json.dumps(body).encode(), headers={**H, **({"Mcp-Session-Id": sid} if sid else {})})
        with urllib.request.urlopen(req, timeout=60) as r:
            sid = r.headers.get("Mcp-Session-Id") or sid
            txt = r.read().decode("utf-8")
        data = [l[5:].strip() for l in txt.splitlines() if l.startswith("data:")]
        msg = json.loads(data[-1]) if data else (json.loads(txt) if txt.strip() else {})
        if "error" in msg:
            raise RuntimeError(msg["error"])
        return sid, msg
    sid, _ = rpc({"jsonrpc": "2.0", "id": 1, "method": "initialize",
                  "params": {"protocolVersion": "2025-03-26", "capabilities": {}, "clientInfo": {"name": "annotator", "version": "0.1"}}})
    rpc({"jsonrpc": "2.0", "method": "notifications/initialized"}, sid)
    _, msg = rpc({"jsonrpc": "2.0", "id": 2, "method": "tools/call",
                  "params": {"name": "web_search_exa", "arguments": {"query": query, "numResults": 5}}}, sid)
    return "\n\n".join(c.get("text", "") for c in msg["result"]["content"])

EXA_COOLDOWN = {"until": 0}
def web_search(query):            # Exa MCP when it isn't rate-limiting us; otherwise DuckDuckGo
    if time.time() > EXA_COOLDOWN["until"]:
        try:
            return exa_mcp(query)[:2500]          # keeps the judge's context (and your credits) small
        except Exception as e:
            EXA_COOLDOWN["until"] = time.time() + 600
            log(f"  Exa MCP failed ({str(e)[:80]}); DuckDuckGo only for the next 10 min")
    for attempt in range(3):
        try:
            hits = DDGS().text(query, max_results=5)
            return "\n\n".join(f"{h.get('title')}\n{h.get('href')}\n{h.get('body')}" for h in hits) or "No results."
        except Exception:
            time.sleep(3 * 2 ** attempt)
    log(f"  SEARCH FAILED: {query[:60]}")
    return "Search failed."

SEARCH = {"name": "web_search", "description": "Search the web. Returns the top results (title, URL, snippet).",
          "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}

def judge(user_prompt):
    """The repo's prompt as one user message; the judge may call web_search up to MAX_SEARCHES times."""
    n, found = 0, []
    tools = [{"type": "function", "function": SEARCH}]
    msgs = [{"role": "user", "content": user_prompt}]
    for _ in range(MAX_SEARCHES + 5):            # tools stay on (Groq rejects a tool call when tools are removed)
        m = client.chat.completions.create(model=JUDGE, messages=msgs, tools=tools, temperature=JUDGE_TEMPERATURE,
                                           max_tokens=JUDGE_MAX_TOKENS).choices[0].message
        if not m.tool_calls:
            return m.content or ""
        msgs.append({"role": "assistant", "content": m.content or "", "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
        for tc in m.tool_calls:
            n += 1
            query = json.loads(tc.function.arguments or "{}").get("query", "")
            result = web_search(query) if n <= MAX_SEARCHES else "Search limit reached. Give your final answer now."
            if n <= MAX_SEARCHES:
                found.append(f"[search: {query}]\n{result}")
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    # rare: it kept searching past the cap -> one last call without tools, with the results it already gathered
    final = (user_prompt + "\n\nWeb search results already gathered:\n\n" + "\n\n".join(found)
             + "\n\nNo more searches are possible. Return the JSON array now.")
    return client.chat.completions.create(model=JUDGE, messages=[{"role": "user", "content": final}],
                                          temperature=JUDGE_TEMPERATURE, max_tokens=JUDGE_MAX_TOKENS).choices[0].message.content or ""

HINDI = ("\n\nNote: the instruction and the completion are in Hindi. Copy every \"span\" exactly as it appears in the completion, "
         "in Devanagari script, without translating or transliterating it. You may search in English or Hindi. "
         "Write the verification_note in English.")
ANNOTATOR = f"hf/{JUDGE}+exa-mcp/ddgs" + ("+hindi-note" if HINDI_NOTE else "")

def annotate(row):
    """Same steps as the repo's annotate_completion (instruction = conversation[-2], completion = conversation[-1])."""
    instruction, completion = row["conversation"][-2]["content"], row["conversation"][-1]["content"]
    prompt = format_prompt(instruction, completion, PROMPT) + (HINDI if HINDI_NOTE else "")
    for attempt in range(3):
        try:
            spans = parse_and_validate_json(judge(prompt), List[AnnotatedSpan], allow_partial=True)
            spans = assign_span_positions(spans, completion)
            log(f"  done {row['prompt_id']}: {len(spans)} entities")
            return {**row, "annotations": [s.model_dump() for s in spans], "annotator": ANNOTATOR}
        except Exception as e:
            log(f"  attempt {attempt + 1} failed for {row['prompt_id']}: {str(e)[:300]}")
    return {**row, "annotations": None, "annotator": None}   # failed; left unlabelled

# every finished answer is appended to PROGRESS at once and uploaded every SAVE_EVERY answers; a re-run skips them
api.create_repo(ANN_REPO, repo_type="dataset", private=False, exist_ok=True)
PROGRESS = f"{WORK}/progress.jsonl"
if not os.path.exists(PROGRESS) and "progress/progress.jsonl" in set(api.list_repo_files(ANN_REPO, repo_type="dataset")):
    shutil.copy(hf_hub_download(ANN_REPO, "progress/progress.jsonl", repo_type="dataset", token=HF_TOKEN), PROGRESS)
saved = {}
if os.path.exists(PROGRESS):
    for l in open(PROGRESS, encoding="utf-8"):
        r = json.loads(l); saved[r["prompt_id"]] = r
lock, since_upload = threading.Lock(), [0]

def push_progress():
    try:
        api.upload_file(path_or_fileobj=PROGRESS, path_in_repo="progress/progress.jsonl", repo_id=ANN_REPO, repo_type="dataset",
                        commit_message=f"progress: {len(saved)} answers")
    except Exception as ex:
        log(f"  progress upload failed (kept locally): {str(ex)[:100]}")

def annotate_and_save(row):
    if row["prompt_id"] in saved:
        return saved[row["prompt_id"]]
    r = annotate(row)
    if r["annotations"] is not None:               # failures are not saved, so they are retried on the next run
        with lock:
            saved[r["prompt_id"]] = r
            with open(PROGRESS, "a", encoding="utf-8") as fh:
                fh.write(json.dumps(r, ensure_ascii=False) + "\n")
            since_upload[0] += 1
            if since_upload[0] >= SAVE_EVERY:
                since_upload[0] = 0
                push_progress()
    return r

log(f"stage 2: {len(answers)} answers, {len(saved)} already annotated -> https://huggingface.co/datasets/{ANN_REPO}")
print("search test:", web_search("Tylenol murders 1982 Chicago")[:200])
with ThreadPoolExecutor(ANN_WORKERS) as pool:
    annotated = list(pool.map(annotate_and_save, answers))
push_progress()
labels = [a["label"] for r in annotated for a in r["annotations"] or [] if a["index"] is not None]
n_failed = sum(r["annotations"] is None for r in annotated)
log(f"stage 2 done: {len(annotated) - n_failed} annotated, {n_failed} failed (run this cell again to retry them), {len(labels)} entities: "
    f"{labels.count('Supported')} S / {labels.count('Not Supported')} NS / {labels.count('Insufficient Information')} II")

In [ ]:
# ---------------- stage 3: model (fp16) + this run's annotated answers ----------------
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer
try:
    tok = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)
except OSError as ex:                       # gated and no access yet: the ungated copy has the same weight files
    log(f"{MODEL} is not accessible ({str(ex)[:80]}); using {MODEL_FALLBACK}")
    MODEL = MODEL_FALLBACK
    tok = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)
NG = torch.cuda.device_count()     # T4 x2: keep ~5 GB free on GPU 0 (embeddings + output layer live there) and ~1.5 GB on GPU 1 for activations
kw = dict(device_map="auto", max_memory={i: ("9GiB" if i == 0 else "13GiB") for i in range(NG)} if NG > 1 else None, token=HF_TOKEN, attn_implementation="sdpa")
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, **kw)        # T4 has no bf16: fp16
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, **kw)
model.eval().requires_grad_(False)
blocks = model.model.layers
NL, D, NH = len(blocks), model.config.hidden_size, model.config.num_attention_heads
HD = D // NH
log(f"{MODEL}: {NL} layers, d_model {D}, {NH} heads")

ours = {}
for l in open(PROGRESS, encoding="utf-8"):
    r = json.loads(l)
    if r.get("annotations") is not None:
        ours[r["prompt_id"]] = {"conversation": r["conversation"], "annotations": r["annotations"], "source": "ours",
                    "id": r["prompt_id"], "turn": 1, "group": r["prompt_id"]}
ours = list(ours.values())
groups = sorted({r["group"] for r in ours}); random.Random(SEED).shuffle(groups)
fold_of = {g: i % FOLDS for i, g in enumerate(groups)}
paper = []
for repo in (["obalcells/longfact-annotations", "obalcells/longfact-augmented-annotations"] if N_PAPER_TRAIN else []):
    ds = load_dataset(repo, "Meta-Llama-3.1-8B-Instruct", split="train", token=HF_TOKEN).shuffle(seed=SEED)
    paper += [{"conversation": r["conversation"], "annotations": r["annotations"], "source": "paper", "id": f"{repo}-{i}", "turn": 1, "group": None}
              for i, r in enumerate(ds.select(range(min(N_PAPER_TRAIN, len(ds)))))]
log(f"ours: {len(ours)} answers in {len(groups)} groups, {FOLDS} folds; paper train texts: {len(paper)}")

In [ ]:
# ---------------- token labels: the paper's rules, applied to the LAST answer ----------------
def chat_ids(msgs, gen=False):
    return tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=gen), add_special_tokens=False)["input_ids"]

def common_prefix(a, b):
    n = 0
    while n < min(len(a), len(b)) and a[n] == b[n]:
        n += 1
    return n

def label_example(row, max_len=MAX_LEN):
    conv = [{"role": m["role"], "content": m["content"]} for m in row["conversation"]]
    completion, full = conv[-1]["content"], chat_ids(conv)
    start = common_prefix(full, chat_ids(conv[:-1], gen=True))              # first token of the latest answer
    q0 = common_prefix(full, chat_ids(conv[:-2])) if len(conv) > 2 else 1   # first token of the latest user message
    ctx = min(64, max_len // 4)                                             # cut from the left; keep some context and BOS
    lo = max(0, len(full) - max_len)
    if lo > start - ctx:
        lo = max(0, start - ctx)
    ids = full[lo:lo + max_len]
    if lo > 0 and tok.bos_token_id is not None:
        ids = [tok.bos_token_id] + ids[1:]
    comp, ids_t = start - lo, torch.tensor(ids)
    labels = torch.full((len(ids),), -100.0); labels[comp:] = 0.0          # paper: answer tokens default 0, prompt ignored
    weights = torch.ones(len(ids))
    pos, neg, spans, cur = [], [], [], comp
    for a in sorted([a for a in row["annotations"] if a.get("index") is not None], key=lambda a: a["index"]):
        y = _MAP_LABEL_TO_SCALAR.get(a.get("label"), -100.0)
        if y == -100.0 or not a["span"] or a["span"] not in completion:
            continue
        try:                                                                # paper: search after the previous span
            s = find_string_in_tokens(a["span"], ids_t[cur:], tok)
        except (AssertionError, ValueError):
            continue
        a0, a1 = s.start + cur, s.stop + cur - 1
        if a0 < comp:
            continue
        cur = a0
        labels[a0:a1 + 1] = y; weights[a0:a1 + 1] = POS_WEIGHT if y == 1.0 else NEG_WEIGHT
        (pos if y == 1.0 else neg).append([a0, a1]); spans.append((a0, a1, y, a["span"], a.get("label")))
    for a0, a1 in pos:                                                      # positives win overlaps (paper)
        labels[a0:a1 + 1] = 1.0; weights[a0:a1 + 1] = POS_WEIGHT
    seg = {"earlier turns": (1, max(1, q0 - lo)), "current question": (max(1, q0 - lo), comp), "current answer": (comp, len(ids))}
    return {"ids": ids_t, "labels": labels, "weights": weights, "pos": pos, "neg": neg, "spans": spans, "comp": comp, "seg": seg, "row": row}

ex = label_example(ours[0])
print(f"{ex['row']['id']}: {len(ex['ids'])} tokens, answer from token {ex['comp']}, {len(ex['pos'])} hallucinated + {len(ex['neg'])} supported entities labelled")
n_ann = sum(1 for r in ours for a in r["annotations"] if a.get("index") is not None and _MAP_LABEL_TO_SCALAR.get(a.get("label"), -100.0) != -100.0)
n_lab = sum(len(label_example(r)["spans"]) for r in ours)
print(f"all answers: {n_lab} of {n_ann} judged entities found in the tokens and used for training / scoring")

In [ ]:
# ---------------- one forward pass: residual stream, attention output, per-head outputs, next-token entropy ----------------
def get_acts(ids):
    got, hooks = {}, []
    for l in range(NL):
        hooks.append(blocks[l].register_forward_hook(lambda m, a, o, l=l: got.__setitem__(("resid", l), (o[0] if isinstance(o, tuple) else o)[0].detach())))
        hooks.append(blocks[l].self_attn.register_forward_hook(lambda m, a, o, l=l: got.__setitem__(("attn", l), (o[0] if isinstance(o, tuple) else o)[0].detach())))
        hooks.append(blocks[l].self_attn.o_proj.register_forward_pre_hook(lambda m, a, l=l: got.__setitem__(("head", l), a[0][0].detach())))
    try:
        with torch.no_grad():
            h = model.model(input_ids=ids[None].to(model.device)).last_hidden_state[0]   # no full-vocab logits (Aya: 256k x seq = 1+ GB)
            W, scale, ent = model.lm_head, getattr(model, "logit_scale", 1.0), []           # Cohere multiplies logits by logit_scale; Llama has none
            for i in range(0, len(h), 256):                                                  # next-token entropy, 256 positions at a time
                p = (W(h[i:i + 256].to(W.weight.device)).float() * scale).log_softmax(-1)
                ent.append(-(p.exp() * p).sum(-1).cpu())
            ent = torch.cat(ent)
            got["entropy"] = torch.cat([torch.zeros(1), ent[:-1]])                          # entropy of the distribution that produced each token
            del h, p
    finally:
        for h in hooks:
            h.remove()
    return got

# ---------------- trainable probes (the paper's loss), one per layer per family ----------------
class HeadProbe(nn.Module):                            # a separate linear probe on each attention head's output
    def __init__(self):
        super().__init__()
        self.w, self.b = nn.Parameter(torch.randn(NH, HD) * 0.01), nn.Parameter(torch.zeros(NH))
    def forward(self, z):                              # z: [seq, NH * HD] (input of o_proj) -> logits [NH, seq]
        return torch.einsum("shd,hd->hs", z.view(len(z), NH, HD), self.w) + self.b[:, None]

def make_probe(fam, l):
    torch.manual_seed(SEED + l)                        # a different start per layer, so cross-layer cosines are not inflated by a shared init
    if fam in ("lin", "attn"):                         # paper: N(0, 0.01) weights, zero bias
        h = nn.Linear(D, 1); nn.init.normal_(h.weight, 0.0, 0.01); nn.init.zeros_(h.bias); return h
    if fam == "mlp":
        return nn.Sequential(nn.Linear(D, MLP_HIDDEN), nn.GELU(), nn.Linear(MLP_HIDDEN, 1))
    return HeadProbe()
SRC = {"lin": "resid", "mlp": "resid", "attn": "attn", "head": "head"}

def probe_logits(fam, h, x):                           # -> [n, seq] (n = 1, or NH for head probes)
    return h(x.float()) if fam == "head" else h(x.float()).T

def probe_loss(logits, e, omega):
    # paper trainer: (1 - omega) * token BCE + omega * span-max BCE, omega rising 0 -> 1 over training
    n = len(logits)
    lab, w = e["labels"].to(logits.device)[None].expand(n, -1), e["weights"].to(logits.device)[None].expand(n, -1)
    bce = compute_probe_bce_loss(logits, lab, w)
    mx = compute_probe_max_aggregation_loss(logits, lab, w, [e["pos"]] * n, [e["neg"]] * n)
    return (1 - omega) * bce + omega * mx

In [ ]:
# ---------------- train + score, fold by fold (checkpointed) ----------------
def ent_mask(e):
    m = torch.zeros(len(e["ids"]), dtype=torch.bool)
    for a0, a1 in e["pos"] + e["neg"]:
        m[a0:a1 + 1] = True
    return m

LR_CS = (1e-4, 1e-3, 1e-2, 1e-1, 1.0)           # L2 strengths tried for logistic regression (smaller C = stronger)
def pick_C(X, Y, G):
    # entity tokens of one answer are near-duplicates, so the strength is chosen by AUC on held-out ANSWERS (grouped CV inside the training fold)
    cv = GroupKFold(n_splits=min(3, len(set(G))))
    sc = {C: np.nanmean(cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(C=C, fit_intercept=False, max_iter=1000)),
                                        X, Y, groups=G, cv=cv, scoring="roc_auc", error_score=np.nan, n_jobs=-1)) for C in LR_CS}
    sc = {C: v for C, v in sc.items() if not np.isnan(v)}
    return max(sc, key=sc.get) if sc else 0.1     # ARENA's default if no split had both classes

def run_fold(f):
    done_file = f"{OUT}/fold{f}_{RUN}.pt"
    if not os.path.exists(done_file):
        try:
            os.replace(hf_hub_download(OUT_REPO, f"{RUN}/fold{f}.pt", token=HF_TOKEN), done_file)
        except Exception:
            pass
    if os.path.exists(done_file):
        log(f"fold {f}: already done"); return torch.load(done_file, weights_only=False)
    train = [r for r in ours if fold_of[r["group"]] != f] + paper
    test = [r for r in ours if fold_of[r["group"]] == f]
    order = [i for ep in range(EPOCHS) for i in random.Random(SEED + ep).sample(range(len(train)), len(train))]
    total = math.ceil(len(order) / BATCH)
    heads = {(fam, l): make_probe(fam, l).to(next(blocks[l].parameters()).device) for fam in SRC for l in range(NL)}
    params = [p for h in heads.values() for p in h.parameters()]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / total))    # linear decay, as HF Trainer
    ck, k0 = f"{OUT}/ckpt_{RUN}_fold{f}.pt", 0
    if os.path.exists(ck):
        c = torch.load(ck, map_location="cpu", weights_only=False)
        k0 = c["k"]; opt.load_state_dict(c["opt"]); sched.load_state_dict(c["sched"])
        for key, h in heads.items():
            h.load_state_dict(c["heads"][f"{key[0]}|{key[1]}"])
        log(f"fold {f}: resuming at text {k0}/{len(order)}")
    t0 = time.time()
    for k in range(k0, len(order)):
        row = train[order[k]]
        e = label_example(row)
        if not e["pos"] and not e["neg"]:
            continue
        acts = get_acts(e["ids"])
        omega = min(1.0, (k // BATCH) / total)
        for (fam, l), h in heads.items():
            (probe_loss(probe_logits(fam, h, acts[SRC[fam], l]), e, omega) / BATCH).backward()
        if (k + 1) % BATCH == 0 or k + 1 == len(order):
            for h in heads.values():
                torch.nn.utils.clip_grad_norm_(h.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
        del acts
        if (k + 1) % CKPT_EVERY == 0:
            torch.save({"k": k + 1, "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "heads": {f"{a}|{b}": h.state_dict() for (a, b), h in heads.items()}}, ck)
            log(f"fold {f}: {k + 1}/{len(order)} texts, ~{(time.time() - t0) / (k + 1 - k0) * (len(order) - k - 1) / 60:.0f} min left")
    # closed-form / sklearn probes (ARENA) on entity tokens of the training answers: all of ours + CACHE_PAPER paper texts
    cache, cache_y, cache_g, arena, lr_C = {l: [] for l in range(NL)}, [], [], {}, {}
    for gi, row in enumerate([r for r in train if r["source"] == "ours"] + [r for r in train if r["source"] == "paper"][:CACHE_PAPER]):
        e = label_example(row); m = ent_mask(e)
        if m.any():
            acts = get_acts(e["ids"])
            for l in range(NL):
                cache[l].append(acts["resid", l][m.to(acts["resid", l].device)].half().cpu())
            cache_y.append(e["labels"][m]); cache_g.append(torch.full((int(m.sum()),), gi)); del acts
    if cache_y:
        Y, G = torch.cat(cache_y), torch.cat(cache_g).numpy()
        for l in range(NL):
            X = torch.cat(cache[l]).float()
            lr_C[l] = pick_C(X.numpy(), Y.numpy(), G)
            arena["LR", l], arena["MM", l] = LRProbe.from_data(X, Y, C=lr_C[l]), MMProbe.from_data(X, Y)
        log(f"fold {f}: LR / MM fitted on {len(Y)} entity tokens from {len(set(G))} answers at every layer; chosen C per layer: {lr_C}")
    # score the held-out answers: per-token probabilities for every detector, span maxima for every head
    ph = None
    if PAPER_PROBE:
        cfg = json.load(open(hf_hub_download("obalcells/hallucination-probes", f"{PAPER_PROBE}/probe_config.json")))
        ph = nn.Linear(cfg["hidden_size"], 1)
        ph.load_state_dict({k: v.float() for k, v in torch.load(hf_hub_download("obalcells/hallucination-probes", f"{PAPER_PROBE}/probe_head.bin"),
                                                                  map_location="cpu", weights_only=True).items()})
    records = []
    for row in test:
        e = label_example(row)
        if not e["pos"] and not e["neg"]:
            continue
        acts = get_acts(e["ids"])
        with torch.no_grad():
            probs, headmax = {}, torch.zeros(NL, NH, len(e["pos"]) + len(e["neg"]))
            for (fam, l), h in heads.items():
                p = torch.sigmoid(probe_logits(fam, h, acts[SRC[fam], l])).float().cpu()
                if fam == "head":
                    headmax[l] = torch.stack([p[:, a0:a1 + 1].max(1).values for a0, a1 in e["pos"] + e["neg"]], 1)
                else:
                    probs[f"{fam}-L{l}"] = p[0]
            for (name, l), pr in arena.items():
                probs[f"{name}-L{l}"] = pr(acts["resid", l].float().cpu())
            probs["entropy"] = acts["entropy"] / math.log(len(tok))          # scaled to [0, 1]
            if ph is not None:
                x = acts["resid", cfg["layer_idx"]].float().cpu()
                probs[f"paper-L{cfg['layer_idx']}"] = torch.sigmoid(ph(x)).squeeze(-1)
        m = ent_mask(e)
        records.append({"e": e, "probs": probs, "headmax": headmax, "ent_acts": {l: acts["resid", l][m.to(acts["resid", l].device)].half().cpu() for l in range(NL)}})
        del acts
    out = {"records": records, "lr_C": lr_C, "heads": {f"{a}|{b}": h.state_dict() for (a, b), h in heads.items() if a in ("lin", "attn")},
           "arena_dirs": {f"{a}|{b}": (p.direction.float().cpu() / (p.scaler_scale if a == "LR" else 1)) for (a, b), p in arena.items()}}
    torch.save(out, done_file)
    try:
        api.create_repo(OUT_REPO, exist_ok=True)
        api.upload_file(path_or_fileobj=done_file, path_in_repo=f"{RUN}/fold{f}.pt", repo_id=OUT_REPO, commit_message=f"{RUN}: fold {f}")
    except Exception as ex:
        log(f"upload failed (kept locally): {str(ex)[:120]}")
    del heads, opt; gc.collect(); torch.cuda.empty_cache()
    return out

folds = [run_fold(f) for f in range(FOLDS)]
records = [r for fo in folds for r in fo["records"]]
log(f"scored {len(records)} held-out answers")

In [ ]:
# ---------------- metrics (the paper's three levels) for every detector ----------------
def metrics(recs, key):
    out, lv = {"n_answers": len(recs)}, {"all": ([], []), "span": ([], []), "span_max": ([], [])}
    for r in recs:
        e, p = r["e"], r["probs"][key]
        m = e["labels"] != -100.0
        lv["all"][0].append(p[m]); lv["all"][1].append(e["labels"][m])
        for y, sp in ((1.0, e["pos"]), (0.0, e["neg"])):
            for a0, a1 in sp:
                lv["span"][0].append(p[a0:a1 + 1]); lv["span"][1].append(torch.full((a1 - a0 + 1,), y))
                lv["span_max"][0].append(p[a0:a1 + 1].max()[None]); lv["span_max"][1].append(torch.tensor([y]))
    for name, (ps, ys) in lv.items():
        pr, yy = torch.cat(ps).numpy(), torch.cat(ys).numpy()
        for mk, mv in compute_clf_metrics((pr > 0.5).astype(float), yy, pr).items():
            out[f"{name}_{mk}"] = mv
    return out

sets = {"all": records}

keys = list(records[0]["probs"])
res = pd.DataFrame([{"detector": k, "family": k.rsplit("-L", 1)[0], "layer": int(k.rsplit("-L", 1)[1]) if "-L" in k else -1, "set": s, **metrics(rs, k)}
                    for k in keys for s, rs in sets.items() if rs])
# per-head span-max AUC (out of fold)
ys = torch.cat([torch.tensor([1.0] * len(r["e"]["pos"]) + [0.0] * len(r["e"]["neg"])) for r in records]).numpy()
hm = torch.cat([r["headmax"] for r in records], 2)
head_auc = np.array([[compute_clf_metrics((hm[l, h].numpy() > 0.5).astype(float), ys, hm[l, h].numpy())["auc"] for h in range(NH)] for l in range(NL)])
res.to_csv(f"{OUT}/results_{RUN}.csv", index=False); np.save(f"{OUT}/head_auc_{RUN}.npy", head_auc)
A = res[res.set == "all"]
BEST_LAYER = int(A[A.family == "lin"].sort_values("span_max_auc").iloc[-1].layer)
best = A.sort_values("span_max_auc", ascending=False).groupby("family").head(1)
COLS = ["span_max_auc", "span_max_recall_at_0.1_fpr", "span_max_f1", "span_auc", "all_auc", "all_recall_at_0.1_fpr", "all_f1"]
print(f"best layer for the linear probe: {BEST_LAYER}")
display(best[["detector"] + COLS].round(3))

In [ ]:
# ---------------- figures (300 dpi, saved to figs/) ----------------
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{OUT}/figs/{RUN}_{name}.png", dpi=300); plt.show()
COL = {"lin": "#d33a2c", "mlp": "#9b72b8", "attn": "#c9a227", "LR": "#2e9b45", "MM": "#4f8a63", "entropy": "#8a8f96", "paper": "#111111", "head": "#6f8fb8"}

# 1. which layer carries the most hallucination information? (span-max AUC per layer, every detector; head = best head of the layer)
fig, ax = plt.subplots(figsize=(8, 4))
for fam in ("lin", "mlp", "attn", "LR", "MM"):
    d = A[A.family == fam].sort_values("layer")
    ax.plot(d.layer, d.span_max_auc, "-o", ms=3, color=COL[fam], label=fam)
ax.plot(range(NL), head_auc.max(1), "--", color=COL["head"], label="best single head")
ax.axhline(A[A.family == "entropy"].span_max_auc.iloc[0], color=COL["entropy"], ls="-.", label="entropy (no training)")
for _, r in A[A.family == "paper"].iterrows():
    ax.plot(r.layer, r.span_max_auc, "*", ms=13, color=COL["paper"], label="paper's released probe")
ax.axhline(0.5, ls=":", color="k", lw=1); ax.set_xlabel("layer"); ax.set_ylabel("span-max AUC (held-out)"); ax.set_ylim(0.3, 1)
ax.legend(fontsize=7, ncol=3); ax.set_title(f"{RUN}: hallucination information by layer"); savefig("layers")

# 2. every paper metric, best layer of each detector
fig, ax = plt.subplots(figsize=(9, 3.8)); w = 0.8 / len(best)
for j, (_, r) in enumerate(best.iterrows()):
    ax.bar(np.arange(len(COLS)) + j * w, [r[c] for c in COLS], w, color=COL[r.family], label=r.detector)
ax.set_xticks(np.arange(len(COLS)) + 0.4 - w / 2); ax.set_xticklabels([c.replace("_", " ") for c in COLS], fontsize=7, rotation=12)
ax.set_ylim(0, 1); ax.legend(fontsize=7, ncol=4); ax.set_title(f"{RUN}: paper metrics, best layer of each detector"); savefig("metrics")

# 4. which attention heads carry it? per-head probe AUC (left) and each head's direct push along the linear probe direction (right)
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4))
im = axs[0].imshow(head_auc, aspect="auto", cmap="viridis", vmin=0.5, vmax=max(0.51, np.nanmax(head_auc)), origin="lower")
axs[0].set_xlabel("head"); axs[0].set_ylabel("layer"); axs[0].set_title("span-max AUC of a probe on each head's output"); plt.colorbar(im, ax=axs[0])
#    direct contribution: head output -> o_proj slice -> dot with the lin probe direction at BEST_LAYER (fold 0), hallucinated minus supported
wdir = folds[0]["heads"][f"lin|{BEST_LAYER}"]["weight"][0].float()
contrib, n = {1: torch.zeros(BEST_LAYER + 1, NH), 0: torch.zeros(BEST_LAYER + 1, NH)}, {1: 0, 0: 0}
for r in [r for r in folds[0]["records"]][:ATTN_EXAMPLES]:
    e = label_example(r["e"]["row"], max_len=ATTN_WINDOW)
    acts = get_acts(e["ids"])
    for l in range(BEST_LAYER + 1):
        Wo = blocks[l].self_attn.o_proj.weight.float()                       # [D, NH * HD]
        z = acts["head", l].float().view(-1, NH, HD)                          # [seq, NH, HD]
        per_head = torch.einsum("shd,ehd->she", z, Wo.view(D, NH, HD).to(z.device)) @ wdir.to(z.device)   # [seq, NH]
        for y, sp in ((1, e["pos"]), (0, e["neg"])):
            for a0, a1 in sp:
                contrib[y][l] += per_head[a0:a1 + 1].mean(0).cpu(); n[y] += 1 if l == 0 else 0
    del acts
diff = contrib[1] / max(1, n[1]) - contrib[0] / max(1, n[0])
v = float(diff.abs().max())
im = axs[1].imshow(diff.numpy(), aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v, origin="lower")
axs[1].set_xlabel("head"); axs[1].set_ylabel("layer"); plt.colorbar(im, ax=axs[1])
axs[1].set_title(f"head → layer-{BEST_LAYER} probe direction\n(hallucinated − supported)"); savefig("heads")
TOP_HEADS = [(int(i // NH), int(i % NH)) for i in torch.argsort(diff.flatten(), descending=True)[:5]]
print("heads that push hardest toward 'hallucinated' (layer, head):", TOP_HEADS)

# 5. PCA of entity tokens at BEST_LAYER (held-out answers) + projection on the probe direction
X = torch.cat([r["ent_acts"][BEST_LAYER] for r in records]).float()
Y = torch.cat([r["e"]["labels"][torch.cat([torch.arange(a0, a1 + 1) for a0, a1 in r["e"]["pos"] + r["e"]["neg"]]).unique()] for r in records])
pcs = get_pca_components(X, 2); Z = (X - X.mean(0)) @ pcs
fig, axs = plt.subplots(1, 2, figsize=(10, 3.9))
for y, c, nme in ((0, "#2e9b45", "Supported"), (1, "#d33a2c", "Not Supported / Insufficient")):
    axs[0].scatter(Z[Y == y, 0], Z[Y == y, 1], s=7, alpha=0.5, color=c, label=nme)
    axs[1].hist((X @ wdir)[Y == y].numpy(), bins=40, alpha=0.6, color=c, density=True, label=nme)
axs[0].set_title(f"PCA of entity tokens, layer {BEST_LAYER}"); axs[0].legend(fontsize=7)
axs[1].set_title(f"projection on the linear probe direction, layer {BEST_LAYER}"); savefig("pca")

# 6. do the detectors find the same direction? (fold 0) — lin across layers, and lin vs attn / LR / MM at BEST_LAYER
dirs = torch.stack([folds[0]["heads"][f"lin|{l}"]["weight"][0].float() for l in range(NL)]); dirs = dirs / dirs.norm(dim=1, keepdim=True)
fig, axs = plt.subplots(1, 2, figsize=(10.5, 4.3))
im = axs[0].imshow((dirs @ dirs.T).numpy(), cmap="RdBu_r", vmin=-1, vmax=1, origin="lower"); plt.colorbar(im, ax=axs[0])
axs[0].set_xlabel("layer"); axs[0].set_ylabel("layer"); axs[0].set_title("cosine of linear-probe directions across layers")
cand = {"lin": folds[0]["heads"][f"lin|{BEST_LAYER}"]["weight"][0].float(), "attn": folds[0]["heads"][f"attn|{BEST_LAYER}"]["weight"][0].float()}
for a in ("LR", "MM"):
    if f"{a}|{BEST_LAYER}" in folds[0]["arena_dirs"]:
        cand[a] = folds[0]["arena_dirs"][f"{a}|{BEST_LAYER}"]
if PAPER_PROBE and BEST_LAYER == 30:
    cand["paper"] = torch.load(hf_hub_download("obalcells/hallucination-probes", f"{PAPER_PROBE}/probe_head.bin"), weights_only=True)["weight"][0].float()
Wc = torch.stack(list(cand.values())); Wc = Wc / Wc.norm(dim=1, keepdim=True)
im = axs[1].imshow((Wc @ Wc.T).numpy(), cmap="RdBu_r", vmin=-1, vmax=1); plt.colorbar(im, ax=axs[1])
axs[1].set_xticks(range(len(cand))); axs[1].set_xticklabels(cand); axs[1].set_yticks(range(len(cand))); axs[1].set_yticklabels(cand)
axs[1].set_title(f"cosine between detectors, layer {BEST_LAYER}"); savefig("directions")

In [ ]:
# ---------------- stage 4 view: what the judge tagged vs what the probe lights up (per word) ----------------
# background = linear-probe score at BEST_LAYER (red above the probe's best span-max threshold, THR), underline = web-search judge label (red = Not Supported / Insufficient, green = Supported).
# Scores are shown per word because one word (and one Devanagari character) can span several tokens.
def word_view(e, probs):
    ids, comp = e["ids"].tolist(), e["comp"]
    ends = [len(tok.decode(ids[comp:j + 1], skip_special_tokens=True)) for j in range(comp, len(ids))]
    text = tok.decode(ids[comp:], skip_special_tokens=True)
    lab = {j - comp: y for a0, a1, y, _, _ in e["spans"] for j in range(a0, a1 + 1)}
    out = []
    for m in re.finditer(r"\S+|\s+", text):
        s0, s1 = m.span()
        if m.group().isspace():
            out.append(htmllib.escape(m.group()).replace("\n", "<br/>")); continue
        tj = [j for j in range(len(ends)) if (ends[j - 1] if j else 0) < s1 and ends[j] > s0]
        sc, ls = max([float(probs[comp + j]) for j in tj], default=0.0), {lab[j] for j in tj if j in lab}
        bg = f"rgba(211,58,44,{0.25 + 0.75 * (sc - THR) / max(1e-6, 1 - THR):.2f})" if sc >= THR else "transparent"
        ul = f"text-decoration:underline;text-decoration-thickness:2.5px;text-decoration-color:{'#d33a2c' if 1.0 in ls else '#2e9b45'};" if ls else ""
        out.append(f'<span title="{sc:.2f}" style="background:{bg};{ul}">{htmllib.escape(m.group())}</span>')
    return "<div style=\"font-family:'Noto Sans Devanagari','Mangal',sans-serif;font-size:15px;line-height:1.9;max-width:900px\">" + "".join(out) + "</div>"

SHOW = f"lin-L{BEST_LAYER}"
THR = float(A[A.detector == SHOW].span_max_optimal_threshold.iloc[0])
print(f"{SHOW}: words above {THR:.3f} (the span-max threshold that maximises accuracy) are shaded")
SHOW_RECS = sorted(records, key=lambda r: -len(r["e"]["pos"]))[:4]
for r in SHOW_RECS:
    e = r["e"]
    page = f"<meta charset='utf-8'><h4>{e['row']['id']} · {SHOW}</h4>" + word_view(e, r["probs"][SHOW])
    open(f"{OUT}/figs/{RUN}_view_{e['row']['id']}.html", "w", encoding="utf-8").write(page); display(HTML(page))
    display(pd.DataFrame([{"entity": s[3], "judge": s[4], "probe max": round(float(r["probs"][SHOW][s[0]:s[1] + 1].max()), 3),
                           "agree": bool(r["probs"][SHOW][s[0]:s[1] + 1].max() >= THR) == (s[2] == 1.0)} for s in e["spans"]]))

In [ ]:
# ---------------- attention: where hallucinated vs supported entity tokens look, by layer + the top head's pattern ----------------
import circuitsvis as cv
try:
    model.set_attn_implementation("eager")             # attention weights need eager attention
except Exception:
    model.config._attn_implementation = "eager"
SEGS = ("current question", "current answer")
mass, cnt = {(y, s): np.zeros(NL) for y in (0, 1) for s in SEGS}, {0: 0, 1: 0}
recs = [r for r in records if r["e"]["row"]["turn"] >= (2 if SETTING == "multiturn" else 1)][:ATTN_EXAMPLES]
for r in recs:
    e = label_example(r["e"]["row"], max_len=ATTN_WINDOW)
    with torch.no_grad():
        att = model.model(input_ids=e["ids"][None].to(model.device), output_attentions=True).attentions   # per layer [1, NH, q, k]
    for y, sp in ((1, e["pos"]), (0, e["neg"])):
        q = [i for a0, a1 in sp for i in range(a0, a1 + 1)]
        if q:
            cnt[y] += 1
            for l, Aw in enumerate(att):
                a = Aw[0].float().mean(0)[q].cpu()
                for s in SEGS:
                    s0, s1 = e["seg"][s]; mass[y, s][l] += a[:, s0:s1].sum(1).mean().item()
    del att; torch.cuda.empty_cache()
fig, ax = plt.subplots(figsize=(7, 3.6))
for (y, s), v in mass.items():
    if cnt[y]:
        ax.plot(v / cnt[y], "-" if y else "--", color={"earlier turns": "#6f8fb8", "current question": "#c9a227", "current answer": "#8a8f96"}[s],
                label=f"{'hallucinated' if y else 'supported'} → {s}")
ax.set_xlabel("layer"); ax.set_ylabel("attention mass (head-averaged)"); ax.legend(fontsize=7); ax.set_title(f"{RUN}: where entity tokens attend"); savefig("attention")

# the attention pattern of the head that pushes hardest toward 'hallucinated', just before the first hallucinated entity
L0, H0 = TOP_HEADS[0]
r = next(r for r in recs if r["e"]["pos"])
e = label_example(r["e"]["row"], max_len=ATTN_WINDOW); a0 = e["pos"][0][0]; win = slice(max(0, a0 - 60), a0 + 10)
with torch.no_grad():
    Aw = model.model(input_ids=e["ids"][None].to(model.device), output_attentions=True).attentions[L0][0][:, win, win].float().cpu()
page = cv.attention.attention_heads(tokens=[tok.decode([i]) for i in e["ids"][win]], attention=Aw, attention_head_names=[f"L{L0}H{h}" for h in range(NH)])
open(f"{OUT}/figs/{RUN}_attention_L{L0}.html", "w", encoding="utf-8").write(str(page)); display(page)
print(f"look at head L{L0}H{H0} in the viewer")
try:
    model.set_attn_implementation("sdpa")
except Exception:
    model.config._attn_implementation = "sdpa"

In [ ]:
# ---------------- stage 5: compare with Hindi on the shared prompts ----------------
# (a) how often the judge flags hallucinations
ann = {THIS: PROGRESS}
for name, repo in COMPARE_ANN.items():
    try:
        ann[name] = hf_hub_download(repo, "progress/progress.jsonl", repo_type="dataset", token=HF_TOKEN)
    except Exception as ex:
        log(f"skipping {name}: {str(ex)[:80]}")
data = {k: {r["prompt_id"]: r for r in map(json.loads, open(v, encoding="utf-8")) if r.get("annotations") is not None} for k, v in ann.items()}
common = sorted(set.intersection(*(set(d) for d in data.values())))
rows = []
for name, d in data.items():
    spans = [[a["label"] for a in d[p]["annotations"] if a["label"] and a["index"] is not None] for p in common]
    flat = [l for s in spans for l in s]
    ns, ii = flat.count("Not Supported"), flat.count("Insufficient Information")
    rows.append({"run": name, "answers": len(common), "entities": len(flat), "entities / answer": len(flat) / max(1, len(common)),
                 "NS %": 100 * ns / max(1, len(flat)), "NS + II %": 100 * (ns + ii) / max(1, len(flat)),
                 "answers with >= 1 NS %": 100 * sum("Not Supported" in s for s in spans) / max(1, len(spans)),
                 "answers with 0 entities %": 100 * sum(not s for s in spans) / max(1, len(spans))})
rates = pd.DataFrame(rows); rates.to_csv(f"{OUT}/judge_rates_{RUN}.csv", index=False)
print(f"judge labels on the {len(common)} prompts every run has ('0 entities' is mostly refusals / very short answers)")
display(rates.round(1))

# (b) how well the probes find them: linear probe by relative depth, and the best layer of every detector
runs = {THIS: res}
for name, r in COMPARE_RUNS.items():
    try:
        runs[name] = pd.read_csv(hf_hub_download(OUT_REPO, f"{r}/results_{r}.csv", token=HF_TOKEN))
    except Exception as ex:
        log(f"skipping {name} probe results: {str(ex)[:80]}")
fig, ax = plt.subplots(figsize=(8, 4))
for (name, d), col in zip(runs.items(), ["#111111", "#d33a2c", "#2e6fb8"]):
    a = d[(d.set == "all") & (d.family == "lin")].sort_values("layer")
    ax.plot(a.layer / a.layer.max(), a.span_max_auc, "-o", ms=3, color=col, label=f"{name}: linear probe")
    for _, p in d[(d.set == "all") & (d.family == "paper")].iterrows():
        ax.plot(p.layer / a.layer.max(), p.span_max_auc, "*", ms=12, color=col, label=f"{name}: paper's English-trained probe")
ax.axhline(0.5, ls=":", color="k", lw=1); ax.set_ylim(0.3, 1); ax.legend(fontsize=7)
ax.set_xlabel("relative depth (layer / last layer)"); ax.set_ylabel("span-max AUC (held-out)"); ax.set_title(f"{THIS} vs the Hindi runs"); savefig("vs_hindi")
best_all = pd.concat([d[d.set == "all"].sort_values("span_max_auc", ascending=False).groupby("family").head(1).assign(run=name) for name, d in runs.items()])
cmp = best_all.pivot_table(index="family", columns="run", values="span_max_auc")
cmp.to_csv(f"{OUT}/probe_compare_{RUN}.csv"); print("best span-max AUC of each detector (its best layer), held-out answers"); display(cmp.round(3))

In [ ]:
# ---------------- upload results + figures ----------------
try:
    api.upload_folder(folder_path=OUT, repo_id=OUT_REPO, path_in_repo=RUN, allow_patterns=["*.csv", "*.npy", "figs/*"], commit_message=f"{RUN}: results, figures")
    log(f"uploaded to https://huggingface.co/{OUT_REPO}/tree/main/{RUN}")
except Exception as ex:
    log(f"upload failed (files are in {OUT}): {ex}")